In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.manifold import TSNE

# Run this notebook from the repository root.
repo_root = Path.cwd().resolve()
if not (repo_root / "analysis/sne_construction").is_dir():
    raise RuntimeError("Run this notebook from the repository root.")
data_dir = repo_root / "data"
analysis_dir = repo_root / "analysis/sne_construction"
overview_dir = analysis_dir / "embedding_overview"
tables_dir = overview_dir / "results/tables"
traits_file = repo_root / "analysis/traits/data/traits_precalculated.txt"

### Embedding t-SNE

In [35]:
# Load both embeddings and retain their shared OTUs.
co_embedding = pd.read_csv(data_dir / "social_niche_embedding_100.txt",
                          header=None, sep=" ", low_memory=False, index_col=0)
co_embedding.drop("<unk>", inplace=True, errors="ignore")
phy_embedding = pd.read_csv(data_dir / "phylo_embed_PCA_100.txt",
                          header=None, sep=" ", low_memory=False, index_col=0)
phy_embedding.drop("<unk>", inplace=True, errors="ignore")
inter_id = np.intersect1d(co_embedding.index, phy_embedding.index)
co_embedding = co_embedding.loc[inter_id]
phy_embedding = phy_embedding.loc[inter_id]

In [36]:
# Project the SNE with cosine-distance t-SNE.
tsne = TSNE(n_components=2, metric="cosine", random_state=42)
result = tsne.fit_transform(co_embedding)

In [37]:
# Save coordinates used by the downstream R plots.
result = pd.DataFrame(data=result, index=co_embedding.index, columns=["t-SNE1", "t-SNE2"])
result.to_csv(tables_dir / "t_sne_co.csv")

In [38]:
# Apply the same projection to the phylogenetic embedding.
tsne = TSNE(n_components=2, metric="cosine", random_state=42)
result = tsne.fit_transform(phy_embedding)
result = pd.DataFrame(data=result, index=phy_embedding.index, columns=["t-SNE1", "t-SNE2"])
result.to_csv(tables_dir / "t_sne_phylo.csv")

### Bugbase

In [39]:
# Restrict both embeddings to OTUs with predicted traits.
df = pd.read_csv(traits_file, sep="\t", index_col=0)

co_embedding_bugbase = co_embedding.loc[df.index.values]
tsne = TSNE(n_components=2, metric="cosine", random_state=42)
result = tsne.fit_transform(co_embedding_bugbase)
result = pd.DataFrame(data=result, index=co_embedding_bugbase.index, columns=["t-SNE1", "t-SNE2"])
result.to_csv(tables_dir / "t_sne_co_bugbase.csv")

phy_embedding_bugbase = phy_embedding.loc[df.index.values]
tsne = TSNE(n_components=2, metric="cosine", random_state=42)
result = tsne.fit_transform(phy_embedding_bugbase)
result = pd.DataFrame(data=result, index=co_embedding_bugbase.index, columns=["t-SNE1", "t-SNE2"])
result.to_csv(tables_dir / "t_sne_phy_bugbase.csv")